# F1 CONSTRUCTOR POINTS

## Análisis Descriptivo

Este proyecto tiene como objetivo desarrollar un dashboard interactivo que permita analizar los puntos promedio obtenidos por cada constructor en cada temporada de Fórmula 1 desde el año 2010 en adelante.

In [62]:
import pandas as pd

In [63]:
constructor_results = pd.read_csv('data/constructor_results.csv')
constructors = pd.read_csv('data/constructors.csv')
races = pd.read_csv('data/races.csv')

In [64]:
constructor_results.head()

,constructorResultsId,raceId,constructorId,points,status
0,1,18,1,14.0,\N
1,2,18,2,8.0,\N
2,3,18,3,9.0,\N
3,4,18,4,5.0,\N
4,5,18,5,2.0,\N


In [65]:
constructor_results.dtypes

constructorResultsId      int64
raceId                    int64
constructorId             int64
points                  float64
status                   object
dtype: object

In [66]:
constructors.head()

,constructorId,constructorRef,name,nationality,url
0,1,mclaren,McLaren,British,http://en.wikipedia.org/wiki/McLaren
1,2,bmw_sauber,BMW Sauber,German,http://en.wikipedia.org/wiki/BMW_Sauber
2,3,williams,Williams,British,http://en.wikipedia.org/wiki/Williams_Grand_Pr...
3,4,renault,Renault,French,http://en.wikipedia.org/wiki/Renault_in_Formul...
4,5,toro_rosso,Toro Rosso,Italian,http://en.wikipedia.org/wiki/Scuderia_Toro_Rosso


In [67]:
races.head()

,raceId,year,round,circuitId,name,date,time,url
0,1,2009,1,1,Australian Grand Prix,29/03/09,6:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...
1,2,2009,2,2,Malaysian Grand Prix,05/04/09,9:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...
2,3,2009,3,17,Chinese Grand Prix,19/04/09,7:00:00,http://en.wikipedia.org/wiki/2009_Chinese_Gran...
3,4,2009,4,3,Bahrain Grand Prix,26/04/09,12:00:00,http://en.wikipedia.org/wiki/2009_Bahrain_Gran...
4,5,2009,5,4,Spanish Grand Prix,10/05/09,12:00:00,http://en.wikipedia.org/wiki/2009_Spanish_Gran...


In [68]:
races.dtypes

raceId        int64
year          int64
round         int64
circuitId     int64
name         object
date         object
time         object
url          object
dtype: object

In [69]:
constructor_results = constructor_results.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
constructor_results.head()

,constructorResultsId,raceId,constructorId,points,status,name
0,1,18,1,14.0,\N,McLaren
1,2,18,2,8.0,\N,BMW Sauber
2,3,18,3,9.0,\N,Williams
3,4,18,4,5.0,\N,Renault
4,5,18,5,2.0,\N,Toro Rosso


In [70]:
constructor_results = constructor_results.merge(races[['raceId', 'year']], on='raceId', how='left')
constructor_results.head()

,constructorResultsId,raceId,constructorId,points,status,name,year
0,1,18,1,14.0,\N,McLaren,2008
1,2,18,2,8.0,\N,BMW Sauber,2008
2,3,18,3,9.0,\N,Williams,2008
3,4,18,4,5.0,\N,Renault,2008
4,5,18,5,2.0,\N,Toro Rosso,2008


In [71]:
constructor_results = constructor_results[(constructor_results['year'] >= 2010) & (constructor_results['year'] <= 2021)]
constructor_results.head()


,constructorResultsId,raceId,constructorId,points,status,name,year
9403,13900,337,6,43.0,\N,Ferrari,2010
9404,13901,337,1,21.0,\N,McLaren,2010
9405,13902,337,9,16.0,\N,Red Bull,2010
9406,13903,337,131,18.0,\N,Mercedes,2010
9407,13904,337,10,2.0,\N,Force India,2010


In [72]:
constructor_results.to_csv('data/results/results_intermediate.csv', index=False)

In [73]:
df = pd.read_csv('data/results/results_intermediate.csv')

Sumar todos los puntos de un mismo constructor en cada temporada y sacar la media y la desviación típica

In [74]:
df = df.drop(columns=['constructorId', 'status', 'raceId', 'constructorResultsId'])
df = df.reset_index(drop=True)
df.head()

,points,name,year
0,43.0,Ferrari,2010
1,21.0,McLaren,2010
2,16.0,Red Bull,2010
3,18.0,Mercedes,2010
4,2.0,Force India,2010


In [75]:
df = df.rename(columns={'name': 'constructor', 'year': 'season'})
df.head()

,points,constructor,season
0,43.0,Ferrari,2010
1,21.0,McLaren,2010
2,16.0,Red Bull,2010
3,18.0,Mercedes,2010
4,2.0,Force India,2010


In [76]:
# Group the dataframe to compute mean and std of points for each constructor and season
df_stats = df.groupby(["constructor", "season"]).points.agg(["mean", "std"]).reset_index()

# Rename the columns as requested
df_stats = df_stats.rename(columns={'constructor': 'Constructor', 'season': 'Season', 'mean': 'Mean', 'std': 'Std'})

# Round mean and std to 4 decimal places
df_stats["Mean"] = df_stats["Mean"].round(4)
df_stats["Std"] = df_stats["Std"].round(4)

# Display the result
df_stats.head()

,Constructor,Season,Mean,Std
0,Alfa Romeo,2019,2.7143,4.9411
1,Alfa Romeo,2020,0.4706,0.9432
2,Alfa Romeo,2021,0.5909,1.2212
3,AlphaTauri,2020,6.2941,6.3518
4,AlphaTauri,2021,6.4545,6.8642


### Explicación del código anterior:

- `df.groupby(["constructor", "season"])`: Agrupa el DataFrame `df` por las columnas `"constructor"` y `"season"`. Esto significa que todas las filas con el mismo constructor y temporada se agrupan juntas.  
- `.points.agg(["mean", "std"])`: Para cada grupo, se calculan dos estadísticas sobre la columna `"points"`:  
  - `"mean"`: La media (promedio) de los puntos obtenidos por el constructor en esa temporada.  
  - `"std"`: La desviación estándar de los puntos obtenidos por el constructor en esa temporada, que mide la variabilidad de los puntos.  
- `.reset_index()`: Convierte los datos agrupados en un DataFrame regular, restableciendo el índice. Las columnas agrupadas (`"constructor"` y `"season"`) se convierten en columnas normales en el DataFrame resultante.  

El DataFrame resultante (`df_stats`) tendrá la siguiente estructura:

| Constructor | Season | Mean | Std |
|-------------|--------|------|-----|
| Nombre del constructor | Año | Media de los puntos | Desviación estándar de los puntos |

In [77]:
df_stats.columns = ['team', 'year', 'mean', 'std']
df_stats.to_csv('data/results/final_results.csv', index=False)

In [79]:
max_row = df_stats.loc[df_stats['mean'].idxmax()]
print(max_row)

team    Mercedes
year        2015
mean        37.0
std      10.5515
Name: 69, dtype: object


## Análisis Predictivo